# Week 9

### Goals
- Let the agent combine memory, search and filtering across several actions.
- Keep the behavior observable and bounded.  

### Tasks
- Extend the loop to retrieve preferences, search candidates, filter and inspect details in sequence.
- Store the trace with tool names, validated inputs and concise observations.
- Set step limits and protect against repeated calls.
- Handle contradictory requests, unknown facts and zero-match searches.
- Ensure feedback is used only when relevant and users can override past preferences.  

### Expected Output
- 09_multi_step_agent.ipynb with several full traces.
- A short list of observed failure modes and remedies.

---
#### Setup

In [2]:
import sys

import pandas as pd

sys.path.append("../")
from src.agent import MODEL, run_agent, print_trace
from src.memory import reset_preferences, get_user_preferences

print("Model:", MODEL)

Model: llama3.1:8b


---


In [3]:
# make sure to start with a clean slate
reset_preferences()

{'user_id': 'test_user_01', 'likes': [], 'dislikes': []}

In [4]:
test_set = [
    ("Recommend a highly rated Sci-Fi movie from 1999.", True),
    ("Comedy from the 90s, no romance.", True),
    ("Something with a twist ending.", True),
    ("An atmospheric sci-fi movie.", True),
    ("I disliked The Matrix because it was too dark.", True),
    ("Recommend a highly rated Sci-Fi movie from 1999.", False),                       # memory now applies
    ("Recommend a highly rated Sci-Fi movie from 1999, ignore my preferences.", False),  # override
    ("Recommend a movie from 1850.", True),                                            # zero match
    ("I want an action movie, but no action.", True),                                  # contradiction
    ("A comedy that is under 90 minutes.", True),                                      # unknown fact
]

In [5]:
runs = []
for prompt, reset in test_set:
    if reset:
        reset_preferences()
    print(f"\n{'=' * 70}\n{prompt}\n{'=' * 70}")
    out = run_agent(prompt)
    runs.append({"prompt": prompt, "out": out})
    print(out["answer"])
    print_trace(out["trace"])
    print(f"{out['seconds']} s")


Recommend a highly rated Sci-Fi movie from 1999.
Primary: The Matrix (1999), ID 2571
  Genres: Action, Sci-Fi, Thriller | mean 4.192 from 278 ratings
Alternative: The Iron Giant (1999), ID 2761
  Genres: Adventure, Animation, Children, Drama, Sci-Fi | mean 4.051 from 39 ratings
 1. interpret model   interpret({}) -> ok: {"intent":"recommend","genre":"Sci-Fi","year_from":1999,"year_to":1999,"mood":""}
 2. search    model   filter_movies({'year_to': '1999', 'exclude_genres': '[]', 'limit': '10', 'genre': 'Sci-Fi', 'year_from': '1999'}) -> ok: 2571 | The Matrix (1999) | Action, Sci-Fi, Thriller | mean 4.192 (n=278)
 3. memory    python  get_user_preferences({}) -> ok: 0 likes, 0 dislikes
 4. rank      python  rank_candidates({'n': 10, 'use_memory': True}) -> ok: 5 ranked
 5. details   python  get_movie_details({'ids': [2571, 2761]}) -> ok: primary and alternative
43.0 s

Comedy from the 90s, no romance.
Primary: Pulp Fiction (1994), ID 296
  Genres: Comedy, Crime, Drama, Thriller | mean 

#### Compact Results

In [6]:
rows = []
for r in runs:
    trace = r["out"]["trace"]
    rows.append({
        "prompt": r["prompt"][:50],
        "tools": " > ".join(e["tool"] for e in trace),
        "blocked/errors": sum(e["status"] != "ok" for e in trace),
        "python fallback": any(e["actor"] == "python" and e["stage"] == "search" for e in trace),
        "movie_ids": r["out"]["movie_ids"],
        "seconds": r["out"]["seconds"],
    })
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

,prompt,tools,blocked/errors,python fallback,movie_ids,seconds
0,Recommend a highly rated Sci-Fi movie from 1999.,interpret > filter_movies > get_user_preferences > rank_candidates > get_mov...,0,False,"[2571, 2761]",43.0
1,"Comedy from the 90s, no romance.",interpret > filter_movies > get_user_preferences > rank_candidates > get_mov...,0,False,"[296, 608]",21.5
2,Something with a twist ending.,interpret > search_descriptions > get_user_preferences > rank_candidates > g...,0,False,"[47, 44665]",19.0
3,An atmospheric sci-fi movie.,interpret > filter_movies > search_descriptions > get_user_preferences > ran...,0,False,"[260, 1196]",23.6
4,I disliked The Matrix because it was too dark.,interpret > search_movies > save_feedback,0,False,[],8.9
5,Recommend a highly rated Sci-Fi movie from 1999.,interpret > filter_movies > get_user_preferences > rank_candidates > get_mov...,0,False,"[2761, 3175]",26.7
6,"Recommend a highly rated Sci-Fi movie from 1999, i",interpret > filter_movies > rank_candidates > get_movie_details,0,False,"[2571, 2761]",21.3
7,Recommend a movie from 1850.,interpret > filter_movies > filter_movies > filter_movies,0,False,[],41.3
8,"I want an action movie, but no action.",interpret,0,False,[],6.8
9,A comedy that is under 90 minutes.,interpret > filter_movies > filter_movies > get_user_preferences > rank_cand...,0,False,"[119145, 152081]",41.1


---
#### Cleanup

In [ ]:
reset_preferences()

---
### Observed failure modes and remedies

**Setup:** llama3.1:8b via Ollama (CPU only, temperature 0, max 256 tokens per call), 10 test prompts, one run each.  
**Result overview:** 0 of 10 runs needed the Python fallback, and 0 of 10 had blocked or failed tool calls.

#### A. Failure modes I handled

| # | What went wrong | Where I saw it | What fixed it |
|---|---|---|---|
| 1 | The agent skipped memory and ranking | Week 7: it called `filter_movies` and answered directly | Python now runs memory, ranking and details in a fixed order. After the Matrix dislike, the Matrix disappeared from the results and the answers show "Memory: -0.40 disliked genres (Sci-Fi)" |
| 2 | Invented movie IDs and false "saved" messages | Week 7: Toy Story (ID 1) was saved instead of The Matrix, and a save was claimed that never happened | The model only extracts the title. Python searches it, resolves the ID, saves, and writes the confirmation from the tool result ("Saved: dislike for The Matrix (1999).") |
| 3 | Junk argument values | The model sends `''`, `0`, `'[]'`, numbers as strings, `"['Romance']"` as a string | The tools treat these as "not provided" or parse them. No tool errors in the final run |
| 4 | Runaway tool calls | One run produced about 22 mostly identical tool calls within 3 model turns (164 s). Earlier, the same prompt did not finish in 18 minutes without a token cap | Token cap (256), maximum 2 tool calls per message, repeat guard, and the search stage stops once candidates exist. The same prompt took 23.6 s in the final run. I did not see it again, but I cannot say how often it happens |
| 5 | Contradictory request | "I want an action movie, but no action." | Python rule check before any tool call. Polite message, 6.8 s, no search |
| 6 | No matches | "Recommend a movie from 1850." | The model retried with wider years, then Python returns a clear message. No crash |
| 7 | Invented facts in answers | Week 7: "popular franchise", invented directors, ratings that no tool returned | The answer is built from catalog fields, matched tags and the preference note, so no invented facts appeared in the 10 answers |

#### B. Failure modes that are still open

| # | Problem | Evidence | Why I think it happens | Remedy (not implemented) |
|---|---|---|---|---|
| 1 | Retrieval evidence is lost in the ranking | "An atmospheric sci-fi movie": the model called both `filter_movies` and `search_descriptions`, but the answer was Star Wars IV and V, with no matching tags | Candidates from both tools are merged and ranked by rating, so famous Sci-Fi films outrank the tag matches. `search_descriptions` also has no genre filter | Add genre and year filters inside `search_descriptions`, then rank by similarity. Or prefer candidates found by both tools |
| 2 | Wrong or missing constraints in the interpretation | "A comedy that is under 90 minutes": the model invented `year_from` and `year_to` = 2020 and did not extract the runtime, so the "runtime not applied" note never appeared | The small model fills schema fields it should leave empty, and nothing checks the values against the user text | Few-shot examples in the prompt, a check that extracted years appear in the user text, show the interpretation in the answer, or a larger model |
| 3 | Silent relaxation of constraints | Same prompt: 2020 returned nothing, the model widened to 2015 to 2020, and the answer does not mention it | The relaxation is the model's decision and is only visible in the trace | Report the arguments of the successful search in the answer, or let Python relax in a fixed order |
| 4 | Crude memory rule | After disliking The Matrix, every Sci-Fi movie gets -0.40 (both picks in the memory run) | Movie-level feedback is turned into genre penalties | Weight by genre frequency or movie similarity, and allow genre-level preferences |
| 5 | Naive zero-match relaxation | 1850: the model tried 1800 and 1700 and found nothing | The model does not know the catalog's year range | Let Python state the available year range and relax in a better order |
| 6 | Tool calls written as plain text (Week 7) | Seen mostly after tool errors in Week 7, 0 of 10 in the final run | Ollama returns the call as text content instead of a tool call | Parse JSON-looking content. The Python fallback would also catch it |

#### C. General observations

- The model's job shrank to two decisions: interpreting the request and choosing search arguments. This is the main reason it works, and also its limit, because the agent no longer plans a sequence. Where the boundary between agent and pipeline should sit is a design question.
- Typical recommend run: 2 model calls (interpretation and search) and 3 Python steps. Time: 19 to 27 s with one search, about 41 s when the model retried, 9 s for feedback, 7 s for a contradiction. Median over all 10 runs: 22.6 s. In Week 7, two model calls took 33 to 72 s.
- The tools hide sloppy model arguments (the trace shows the raw values the model sent, for example `'1999'` as a string), so tool errors alone would understate model mistakes.
- Retrieval and genre filtering complement each other, but combining them is not solved (B1).
- The results are limited by the data. MovieLens genre labels are coarse (Pulp Fiction and Fargo count as Comedy), and only 12.5% of movies have tags.
- The first prompt of the run took 43.0 s versus 20.9 s earlier. I assume model loading, but I did not verify it.

#### D. Limits of this evaluation

- One local model, 10 prompts, one run each, temperature 0. This shows that failure modes exist, not how often they occur. Frequencies and constraint checks follow in Week 10.
- Comparisons with Week 7 are not controlled, because code and prompts changed together.